# Toxicity Detection & Social Media Analysis

## Notebook 05 – Jigsaw Gender/Race Keyword Validation & Toxicity Inference

### Objectives

- Build ground-truth gender/race flags from Jigsaw's own identity annotation columns (already present in `jigsaw_processed.parquet` — nothing was dropped in Notebook 03).
- Apply the client-provided keyword lists (`gender.txt`, `race.txt`) to the same text.
- Validate the keyword-based method against the annotation-based ground truth (precision/recall), since Bluesky has **no** identity annotations and will rely on keywords alone.
- Run the trained DeBERTa-v3-base model to get toxicity scores.
- Compare mean toxicity across gender-related / race-related / neither, using both grouping methods.
- Save the combined result as `jigsaw_gender_race_validation.parquet`.

### Confirmed from `001_full_data_audit.ipynb` (run 2025, uploaded)

- `train_valid_split/train.parquet` (80,000 rows) and `valid.parquet` (20,000 rows) **only have two columns: `clean_text` and `is_toxic`. No `id` column.** So the join back to `jigsaw_processed.parquet` (for identity columns and keyword flags) has to be on `clean_text`, not `id`.
- `models/deberta_v3_base/best_model/` has only `tokenizer.json` (no `spm.model`/`vocab.txt`/`merges.txt`) — fast tokenizer only, confirmed.
- `reports/eval_metrics.json` confirms the numbers already sent to the client: F1 = 0.8982, ROC-AUC = 0.9611.
- `models/deberta_v3_base/predictions/validation_predictions.csv` already has `true_label`, `predicted_label`, `predicted_probability_toxic` for the validation set — no text column, but since it came straight from `trainer.predict(valid_dataset)` with no shuffling, it should be in the same row order as `valid.parquet`. This notebook re-runs inference independently (needed anyway, since the goal here is new columns joined onto the full row, not just the three saved back in Notebook 04) but cross-checks against this file positionally as a sanity check.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import precision_recall_fscore_support, classification_report

from tqdm.auto import tqdm
tqdm.pandas()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)


d:\Projects\sentiment-analysis-socialmedia\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ------------------------------------------------------------------
# Paths — matched to the actual local folder structure and to what
# Notebook 00 (data_audit) and Notebook 04 (training) confirm about
# column names / tokenizer files. See the checks printed below.
# ------------------------------------------------------------------
ROOT = Path("..")

DATASET_DIR = ROOT / "Dataset"
PROCESSED_DIR = DATASET_DIR / "processed"
TRAIN_VALID_SPLIT_DIR = PROCESSED_DIR / "train_valid_split"

JIGSAW_PROCESSED_PATH = PROCESSED_DIR / "jigsaw_processed.parquet"
JIGSAW_BALANCED_PATH = PROCESSED_DIR / "jigsaw_balanced.parquet"
TRAIN_SPLIT_PATH = TRAIN_VALID_SPLIT_DIR / "train.parquet"
VALID_SPLIT_PATH = TRAIN_VALID_SPLIT_DIR / "valid.parquet"

# Trained model saved in Notebook 04 (best_model/ containing config, weights, tokenizer)
MODEL_DIR = ROOT / "models" / "deberta_v3_base" / "best_model"

# Client-provided keyword lists
KEYWORDS_DIR = DATASET_DIR / "keywords"
GENDER_KEYWORDS_PATH = KEYWORDS_DIR / "gender.txt"
RACE_KEYWORDS_PATH = KEYWORDS_DIR / "race.txt"

OUTPUT_PATH = PROCESSED_DIR / "jigsaw_gender_race_validation.parquet"

MAX_LENGTH = 384
INFERENCE_BATCH_SIZE = 64

print("Jigsaw processed   :", JIGSAW_PROCESSED_PATH, JIGSAW_PROCESSED_PATH.exists())
print("Jigsaw balanced    :", JIGSAW_BALANCED_PATH, JIGSAW_BALANCED_PATH.exists())
print("Train split        :", TRAIN_SPLIT_PATH, TRAIN_SPLIT_PATH.exists())
print("Valid split        :", VALID_SPLIT_PATH, VALID_SPLIT_PATH.exists())
print("Model dir          :", MODEL_DIR, MODEL_DIR.exists())
print("Gender keywords    :", GENDER_KEYWORDS_PATH, GENDER_KEYWORDS_PATH.exists())
print("Race keywords      :", RACE_KEYWORDS_PATH, RACE_KEYWORDS_PATH.exists())


Jigsaw processed   : ..\Dataset\processed\jigsaw_processed.parquet True
Jigsaw balanced    : ..\Dataset\processed\jigsaw_balanced.parquet True
Train split        : ..\Dataset\processed\train_valid_split\train.parquet True
Valid split        : ..\Dataset\processed\train_valid_split\valid.parquet True
Model dir          : ..\models\deberta_v3_base\best_model True
Gender keywords    : ..\Dataset\keywords\gender.txt True
Race keywords      : ..\Dataset\keywords\race.txt True


### Tokenizer file check

`models/deberta_v3_base/best_model/` (per your folder listing) contains `tokenizer.json` but not `spm.model` / `vocab.txt` / `merges.txt`. That means only the **fast** tokenizer was saved, even though Notebook 04 loaded the original pretrained tokenizer with `use_fast=False`. So this notebook loads the tokenizer with `AutoTokenizer.from_pretrained(MODEL_DIR)` (no `use_fast=False`) — forcing the slow tokenizer here would fail since its files aren't present. The cell below re-verifies this at runtime rather than assuming it.

In [3]:
# Runtime check — confirms the tokenizer-file situation before we load anything
if MODEL_DIR.exists():
    present = {f.name for f in MODEL_DIR.glob("*")}
    fast_files = [f for f in ["tokenizer.json"] if f in present]
    slow_files = [f for f in ["spm.model", "vocab.txt", "merges.txt"] if f in present]

    print("Files in best_model/:", sorted(present))
    print("Fast-tokenizer files present:", fast_files or "NONE")
    print("Slow-tokenizer files present:", slow_files or "NONE")

    USE_FAST_TOKENIZER = bool(fast_files) and not slow_files
    print("\n--> Loading with use_fast =", USE_FAST_TOKENIZER)
else:
    raise FileNotFoundError(f"MODEL_DIR not found: {MODEL_DIR}. Update the path in the config cell above.")


Files in best_model/: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']
Fast-tokenizer files present: ['tokenizer.json']
Slow-tokenizer files present: NONE

--> Loading with use_fast = True


## Load Processed Jigsaw Data

In [4]:
jigsaw = pd.read_parquet(JIGSAW_PROCESSED_PATH)

print("Shape:", jigsaw.shape)
jigsaw.head(3)


Shape: (1780822, 47)


,id,target,comment_text,severe_toxicity,obscene,identity_attack,insult,threat,asian,atheist,bisexual,black,buddhist,christian,female,heterosexual,hindu,homosexual_gay_or_lesbian,intellectual_or_learning_disability,jewish,latino,male,muslim,other_disability,other_gender,other_race_or_ethnicity,other_religion,other_sexual_orientation,physical_disability,psychiatric_or_mental_illness,transgender,white,created_date,publication_id,parent_id,article_id,rating,funny,wow,sad,likes,disagree,sexual_explicit,identity_annotator_count,toxicity_annotator_count,clean_text,is_toxic
0,59848,0.0,"This is so cool. It's like, 'would you want your mother to read this??' Really great idea, well done!",0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-29 10:50:41.987077+00,2,NaN,2006,rejected,0,0,0,0,0,0.0,0,4,"this is so cool. it's like, 'would you want your mother to read this??' really great idea, well done!",0
1,59849,0.0,"Thank you!! This would make my life a lot less anxiety-inducing. Keep it up, and don't let anyone get in your way!",0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-29 10:50:42.870083+00,2,NaN,2006,rejected,0,0,0,0,0,0.0,0,4,"thank you!! this would make my life a lot less anxiety-inducing. keep it up, and don't let anyone get in your way!",0
2,59852,0.0,This is such an urgent design problem; kudos to you for taking it on. Very impressive!,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2015-09-29 10:50:45.222647+00,2,NaN,2006,rejected,0,0,0,0,0,0.0,0,4,this is such an urgent design problem; kudos to you for taking it on. very impressive!,0


## Identity Annotation Ground Truth

Jigsaw's annotators scored each comment for how strongly it mentions each identity group (0–1, NaN where no identity mention was annotated at all). We threshold at `>= 0.5`, the same convention used for `is_toxic` in Notebook 03.

Column groupings:

- **Gender**: `female`, `male`, `transgender`, `other_gender`
- **Race / ethnicity**: `black`, `white`, `asian`, `latino`, `other_race_or_ethnicity`

Sexual orientation columns (`homosexual_gay_or_lesbian`, `bisexual`, `heterosexual`, `other_sexual_orientation`) are intentionally excluded — the client asked for gender and race only. Adjust the lists below if that scope changes.

In [5]:
GENDER_ANNOT_COLS = ["female", "male", "transgender", "other_gender"]
RACE_ANNOT_COLS = ["black", "white", "asian", "latino", "other_race_or_ethnicity"]

missing_cols = [c for c in GENDER_ANNOT_COLS + RACE_ANNOT_COLS if c not in jigsaw.columns]
assert not missing_cols, f"Missing expected identity columns: {missing_cols}"

jigsaw["gender_related_annot"] = (jigsaw[GENDER_ANNOT_COLS].fillna(0) >= 0.5).any(axis=1).astype(int)
jigsaw["race_related_annot"] = (jigsaw[RACE_ANNOT_COLS].fillna(0) >= 0.5).any(axis=1).astype(int)

jigsaw[["gender_related_annot", "race_related_annot"]].mean()


gender_related_annot    0.045122
race_related_annot      0.021721
dtype: float64

In [6]:
print("Gender-related (annotation-based):")
print(jigsaw["gender_related_annot"].value_counts())

print()
print("Race-related (annotation-based):")
print(jigsaw["race_related_annot"].value_counts())

print()
print("Overlap (both gender and race related):")
print(((jigsaw["gender_related_annot"] == 1) & (jigsaw["race_related_annot"] == 1)).sum())


Gender-related (annotation-based):
gender_related_annot
0    1700467
1      80355
Name: count, dtype: int64

Race-related (annotation-based):
race_related_annot
0    1742141
1      38681
Name: count, dtype: int64

Overlap (both gender and race related):
9196


## Keyword-Based Detection (Client-Provided Lists)

This replicates the *only* signal available for the Bluesky dataset in the next notebook, so it needs to be validated here against the real annotation labels above.

In [7]:
def load_keywords(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip().lower() for line in f]
    return [line for line in lines if line]


def build_pattern(keywords):
    escaped = [re.escape(kw) for kw in keywords]
    pattern = r"\b(?:" + "|".join(escaped) + r")\b"
    return re.compile(pattern, flags=re.IGNORECASE)


In [8]:
gender_keywords = load_keywords(GENDER_KEYWORDS_PATH)
race_keywords = load_keywords(RACE_KEYWORDS_PATH)

print(f"Gender keywords ({len(gender_keywords)}):", gender_keywords)
print(f"Race keywords ({len(race_keywords)}):", race_keywords)

gender_pattern = build_pattern(gender_keywords)
race_pattern = build_pattern(race_keywords)


Gender keywords (37): ['gender', 'sex', 'male', 'female', 'man', 'men', 'woman', 'women', 'boy', 'boys', 'girl', 'girls', 'guy', 'guys', 'lady', 'ladies', 'gentleman', 'gentlemen', 'transgender', 'trans', 'cisgender', 'cis', 'nonbinary', 'non-binary', 'genderqueer', 'genderfluid', 'agender', 'intersex', 'he', 'him', 'his', 'she', 'her', 'hers', 'they', 'them', 'theirs']
Race keywords (40): ['race', 'ethnicity', 'ethnic', 'minority', 'majority', 'immigrant', 'migrant', 'refugee', 'black', 'white', 'asian', 'african', 'african american', 'caucasian', 'hispanic', 'latino', 'latina', 'latinx', 'indigenous', 'native american', 'first nations', 'pacific islander', 'middle eastern', 'arab', 'south asian', 'east asian', 'southeast asian', 'european', 'chinese', 'japanese', 'korean', 'indian', 'pakistani', 'bangladeshi', 'nigerian', 'mexican', 'irish', 'british', 'french', 'german']


In [9]:
# Vectorised regex matching — fast even at ~1.8M rows
jigsaw["gender_related_kw"] = jigsaw["clean_text"].str.contains(gender_pattern, regex=True, na=False).astype(int)
jigsaw["race_related_kw"] = jigsaw["clean_text"].str.contains(race_pattern, regex=True, na=False).astype(int)

print("Gender-related (keyword-based):")
print(jigsaw["gender_related_kw"].value_counts())

print()
print("Race-related (keyword-based):")
print(jigsaw["race_related_kw"].value_counts())


Gender-related (keyword-based):
gender_related_kw
0    941772
1    839050
Name: count, dtype: int64

Race-related (keyword-based):
race_related_kw
0    1645257
1     135565
Name: count, dtype: int64


## Validating the Keyword Lists Against Annotation Ground Truth

This is the key check before trusting the same keyword lists on Bluesky, which has no annotations to fall back on. Precision here means "when the keyword list flags a post, how often is it actually gender/race-related per the annotators." Recall means "of the posts annotators marked gender/race-related, how many did the keyword list catch."

In [10]:
def evaluate_keyword_method(y_true, y_pred, label):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    print(f"--- {label} ---")
    print(classification_report(y_true, y_pred, target_names=["not_related", "related"], digits=4, zero_division=0))
    return {"category": label, "precision": precision, "recall": recall, "f1": f1}


results = []
results.append(evaluate_keyword_method(jigsaw["gender_related_annot"], jigsaw["gender_related_kw"], "Gender"))
results.append(evaluate_keyword_method(jigsaw["race_related_annot"], jigsaw["race_related_kw"], "Race"))

keyword_validation_summary = pd.DataFrame(results)
keyword_validation_summary


--- Gender ---
              precision    recall  f1-score   support

 not_related     0.9985    0.5530    0.7118   1700467
     related     0.0941    0.9825    0.1717     80355

    accuracy                         0.5724   1780822
   macro avg     0.5463    0.7678    0.4418   1780822
weighted avg     0.9577    0.5724    0.6874   1780822

--- Race ---
              precision    recall  f1-score   support

 not_related     0.9982    0.9427    0.9696   1742141
     related     0.2631    0.9222    0.4094     38681

    accuracy                         0.9422   1780822
   macro avg     0.6306    0.9324    0.6895   1780822
weighted avg     0.9822    0.9422    0.9575   1780822



,category,precision,recall,f1
0,Gender,0.094096,0.982528,0.171744
1,Race,0.263128,0.922184,0.409433


### Observations

_Record notes here after reviewing the precision/recall numbers above — e.g. whether the client's keyword lists need expanding before running on Bluesky, and which category (gender vs race) is weaker._

## Toxicity Inference

Loads the trained model and scores a subset of Jigsaw. See the caveat at the top of this notebook about which rows are used.

In [11]:
if not VALID_SPLIT_PATH.exists():
    raise FileNotFoundError(
        f"{VALID_SPLIT_PATH} not found. This file was confirmed present during the audit — "
        "check the path/config cell above if this fails."
    )

valid_split = pd.read_parquet(VALID_SPLIT_PATH)
print(f"Loaded {VALID_SPLIT_PATH}: {len(valid_split):,} rows")
print("Columns:", valid_split.columns.tolist())

# Confirmed via 001_full_data_audit.ipynb: valid.parquet has only clean_text + is_toxic,
# no id column. Join on clean_text — check for duplicates first since a collision would
# silently multiply rows in the merge below.
dupe_count = valid_split["clean_text"].duplicated().sum()
if dupe_count:
    print(f"NOTE: {dupe_count} duplicate clean_text value(s) within valid.parquet itself.")

jigsaw_dupe_count = jigsaw["clean_text"].duplicated().sum()
print(f"Duplicate clean_text values in jigsaw_processed: {jigsaw_dupe_count:,} (out of {len(jigsaw):,})")

inference_df = jigsaw.merge(
    valid_split[["clean_text", "is_toxic"]].drop_duplicates(subset="clean_text"),
    on="clean_text",
    how="inner",
    suffixes=("", "_valid"),
).reset_index(drop=True)

print(f"Matched {len(inference_df):,} rows out of {len(valid_split):,} in valid.parquet.")

if len(inference_df) != len(valid_split):
    print(
        "WARNING: match count differs from valid.parquet's row count. "
        "Likely cause: a clean_text value matched more than one jigsaw_processed row, "
        "or a clean_text value in valid.parquet has no exact match in jigsaw_processed "
        "(e.g. if jigsaw_processed was regenerated since the split was made). "
        "Investigate before trusting the toxicity numbers below."
    )

# Sanity check: is_toxic from jigsaw_processed should agree with is_toxic from valid.parquet
# for every matched row, since both were derived the same way from the same target column.
mismatch = (inference_df["is_toxic"] != inference_df["is_toxic_valid"]).sum()
print(f"is_toxic mismatches between jigsaw_processed and valid.parquet on matched rows: {mismatch}")


Loaded ..\Dataset\processed\train_valid_split\valid.parquet: 20,000 rows
Columns: ['clean_text', 'is_toxic']
NOTE: 13 duplicate clean_text value(s) within valid.parquet itself.
Duplicate clean_text values in jigsaw_processed: 6,603 (out of 1,780,822)
Matched 21,954 rows out of 20,000 in valid.parquet.
is_toxic mismatches between jigsaw_processed and valid.parquet on matched rows: 17


## Load Trained Model & Tokenizer

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=USE_FAST_TOKENIZER)
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
model.to(device)
model.eval()

print("Tokenizer type:", type(tokenizer).__name__)
print("Model loaded on:", device)


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 2897.35it/s]

Tokenizer type: DebertaV2Tokenizer
Model loaded on: cpu


In [13]:
def predict_toxicity(texts, batch_size=INFERENCE_BATCH_SIZE, max_length=MAX_LENGTH):
    probs = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Running inference"):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            list(batch),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            logits = model(**encoded).logits

        batch_probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        probs.extend(batch_probs.tolist())

    return np.array(probs)


texts = inference_df["clean_text"].fillna("").tolist()
inference_df["pred_prob"] = predict_toxicity(texts)
inference_df["pred_label"] = (inference_df["pred_prob"] >= 0.5).astype(int)

inference_df[["clean_text", "is_toxic", "pred_prob", "pred_label"]].head(10)


Running inference:   1%|          | 3/344 [01:27<2:45:01, 29.04s/it]


KeyboardInterrupt: 

### Cross-check against Notebook 04's saved predictions

`models/deberta_v3_base/predictions/validation_predictions.csv` already has predictions for this exact validation set, saved straight from `trainer.predict(valid_dataset)` — no shuffling in eval, so its row order should match `valid.parquet`'s row order. It has no text column to join on, so this check is positional: if the merge above didn't change row order (it doesn't — `merge` with `how="inner"` on a not-otherwise-sorted key preserves left-frame order here since every `valid_split` row matches at most once), row *i* of this file should equal row *i* of `inference_df`. This confirms the re-run inference above reproduces Notebook 04's original results rather than silently diverging (different padding, truncation, or checkpoint).

In [ ]:
saved_predictions_path = MODELS_DIR / "deberta_v3_base" / "predictions" / "validation_predictions.csv"

if saved_predictions_path.exists() and len(inference_df) == len(valid_split):
    saved_predictions = pd.read_csv(saved_predictions_path)
    print(f"Loaded {saved_predictions_path}: {len(saved_predictions):,} rows")

    if len(saved_predictions) == len(inference_df):
        label_agreement = (saved_predictions["predicted_label"].values == inference_df["pred_label"].values).mean()
        prob_diff = np.abs(saved_predictions["predicted_probability_toxic"].values - inference_df["pred_prob"].values)

        print(f"Predicted-label agreement (positional): {label_agreement:.4%}")
        print(f"Mean |probability difference|: {prob_diff.mean():.6f}")
        print(f"Max |probability difference|:  {prob_diff.max():.6f}")

        if label_agreement < 0.99:
            print("\nWARNING: low agreement — investigate before trusting the re-run inference above.")
    else:
        print(
            f"Row count mismatch ({len(saved_predictions):,} saved vs {len(inference_df):,} here) — "
            "skipping positional comparison, can't align safely."
        )
else:
    print("Saved predictions file not found, or row counts didn't align cleanly above — skipping cross-check.")


## Toxicity by Group — Annotation-Based vs Keyword-Based

In [ ]:
def group_summary(df, gender_col, race_col, label):
    def bucket(row):
        if row[gender_col] and row[race_col]:
            return "gender_and_race"
        if row[gender_col]:
            return "gender_only"
        if row[race_col]:
            return "race_only"
        return "neither"

    grouped = df.assign(group=df.apply(bucket, axis=1)).groupby("group")["pred_prob"].agg(["mean", "count"])
    print(f"--- {label} ---")
    print(grouped)
    print()
    return grouped


annot_summary = group_summary(inference_df, "gender_related_annot", "race_related_annot", "Annotation-based grouping")
kw_summary = group_summary(inference_df, "gender_related_kw", "race_related_kw", "Keyword-based grouping")


## Save Final Validation Dataset

In [ ]:
final_cols = [
    "id", "comment_text", "clean_text", "target", "is_toxic",
    "gender_related_annot", "race_related_annot",
    "gender_related_kw", "race_related_kw",
    "pred_prob", "pred_label",
] + GENDER_ANNOT_COLS + RACE_ANNOT_COLS

final_cols = [c for c in final_cols if c in inference_df.columns]

result = inference_df[final_cols].copy()

PROCESSED_DIR.mkdir(exist_ok=True, parents=True)
result.to_parquet(OUTPUT_PATH, index=False)

print("Saved!")
print(OUTPUT_PATH)
print("Shape:", result.shape)
